# Evaluation of the models

The model outputs are compared against the manually created reference answers using three metrics: ROUGE, BLEU and BERTScore.



In [ ]:
# Cell 1
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Cell 2 : install the pakets for the evaluation
!pip install rouge-score bert-score sacrebleu -q

In [ ]:
# Cell 3: read in the vsv files and merge them
import pandas as pd
import csv

pfad = "/content/drive/MyDrive/Colab Notebooks/ABGABE/"

# correct answer (from the students)
ref_df = pd.read_csv(pfad + "answers_clean.csv", sep=";", engine="python", on_bad_lines="skip")
ref_df = ref_df[["id", "correct_answer"]]

# fine tuning and RAG answers
ft_df  = pd.read_csv(pfad + "FT_output.csv", on_bad_lines="skip", engine="python")
rag_df = pd.read_csv(pfad + "RAG_output_submission.csv", on_bad_lines="skip", engine="python")

# inference in combination with csv.reader because the csv unfortunately was slightly broken and not usable like the others
rows = []
with open(pfad + "inference_final.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f, quotechar='"')
    for row in reader:
        if len(row) == 2:
            rows.append(row)
inf_df = pd.DataFrame(rows[1:], columns=rows[0])

# rename the columns
ft_df  = ft_df.rename(columns={"answer": "answer_ft"})
rag_df = rag_df.rename(columns={"answer": "answer_rag"})
inf_df = inf_df.rename(columns={"answer": "answer_inference"})

# merge
df = ref_df.merge(ft_df, on="id") \
           .merge(rag_df, on="id") \
           .merge(inf_df, on="id")
df = df.dropna()

print(f"Zeilen: {len(df)}")
print(f"Spalten: {df.columns.tolist()}")

Zeilen: 643
Spalten: ['id', 'correct_answer', 'answer_ft', 'answer_rag', 'answer_inference']


In [ ]:
# Cell 4: evaluation
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from sacrebleu.metrics import BLEU

references       = df["correct_answer"].tolist()
predictions_ft   = df["answer_ft"].tolist()
predictions_rag  = df["answer_rag"].tolist()
predictions_inf  = df["answer_inference"].tolist()

def evaluate(predictions, references, name):
    # ROUGE
    scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
    r1, rL = [], []
    for pred, ref in zip(predictions, references):
        s = scorer.score(str(ref), str(pred))
        r1.append(s["rouge1"].fmeasure)
        rL.append(s["rougeL"].fmeasure)

    # BERTScore
    P, R, F1 = bert_score(predictions, references, lang="de")

    # BLEU
    bleu = BLEU()
    bleu_score = bleu.corpus_score(predictions, [references])

    print(f"\n=== {name} ===")
    print(f"ROUGE-1:   {sum(r1)/len(r1):.4f}")
    print(f"ROUGE-L:   {sum(rL)/len(rL):.4f}")
    print(f"BERTScore: {F1.mean():.4f}")
    print(f"BLEU:      {bleu_score.score:.4f}")

evaluate(predictions_ft,  references, "Fine-tuned")
evaluate(predictions_rag, references, "RAG")
evaluate(predictions_inf, references, "Inference")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== RAG ===
ROUGE-1:   0.2258
ROUGE-L:   0.1674
BERTScore: 0.7132
BLEU:      5.0969


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== Inference ===
ROUGE-1:   0.3329
ROUGE-L:   0.2296
BERTScore: 0.7386
BLEU:      9.6853
